# CRTi: count-matrix preprocessing

Source-derived excerpts from the Python notebooks used for my colorectal-cancer single-cell analysis. This notebook shows 10x input handling, barcode metadata and quality-control metrics. Private inputs, sample-to-condition assignments, gene selections and stored results have been removed. The method calls and plotting settings are retained; a small path configuration and input checks make the public copy readable independently.

Run from the repository root with a 10x `filtered_feature_bc_matrix` directory under `data/private/crti/`. It requires Scanpy, NumPy, pandas and Matplotlib. The public notebook has been checked for Python syntax and notebook structure, but has not been run against the withheld data. Numerical settings below are analysis settings, not universal QC recommendations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib import rcParams

# Run from the repository root. These local inputs and outputs are not distributed.
PRIVATE_DIR = Path("data/private/crti")
OUTPUT_DIR = Path("outputs/crti/python")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = OUTPUT_DIR
sc.settings.verbosity = 3
sc.settings.set_figure_params(scanpy=True, dpi=80, dpi_save=300,
                             frameon=True, vector_friendly=True, fontsize=14)
rcParams['figure.figsize'] = 8, 8

In [ ]:
adata = sc.read_10x_mtx(
    PRIVATE_DIR / 'filtered_feature_bc_matrix',
    var_names='gene_symbols',
    cache=False,
)
adata.var_names_make_unique()

## Barcode metadata

The working notebook recovered a library suffix from each 10x barcode and then assigned study sample labels. Here the suffix is retained as a neutral grouping field; the original sample and treatment lookup is withheld. A suffix can identify a capture only when the upstream merged matrix was constructed that way. For other inputs, supply sample metadata from a private manifest instead.

In [ ]:
obs = adata.obs
UMI = obs.index.tolist()
Samples = []
for x in UMI:
    # rsplit supports suffixes longer than one digit.
    suffix = x.rsplit('-', 1)[-1]
    if '-' not in x or not suffix.isdigit():
        raise ValueError('Expected a 10x barcode with a numeric library suffix.')
    Samples.append('Library_' + suffix)
obs['Samples'] = pd.Categorical(Samples)
obs['Barcode'] = obs.index
adata.obs = obs

## Initial filtering and quality-control metrics

The `mt-` prefix is used by this workflow. Check the annotation convention in any new dataset before calculating the mitochondrial fraction.

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=10)

In [ ]:
mito_genes = adata.var_names.str.startswith('mt-')
total_counts = np.asarray(adata.X.sum(axis=1)).ravel()
mito_counts = np.asarray(adata[:, mito_genes].X.sum(axis=1)).ravel()
adata.obs['percent_mito'] = np.divide(
    mito_counts, total_counts,
    out=np.zeros_like(total_counts, dtype=float), where=total_counts > 0,
)
adata.obs['n_counts'] = total_counts

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=100, ) #Computes, for each gene, the fraction of counts assigned to that gene within a cell. 
#The n_top genes with the highest mean fraction over all cells are plotted as boxplots.

In [ ]:
adata.write(OUTPUT_DIR / 'preprocessed-counts.h5ad')